In [ ]:
### Script to integrate, preprocess and harmonize all available data sets
### In our use-case
# Single Cell RNA Seq
# Cytokine Data
# Neutrophil Data
# Clinical Data
# Proteomics

#############################################
# Prerequisites - Load Libraries

In [ ]:
source('MS1_Functions.r')

In [ ]:
### Inform about execution start
popup_function_pos('02_Integrate_and_Normalize_Data_Sources: Execution Started')

In [ ]:
source('MS0_Libraries.r')

In [ ]:
source('MS2_Plot_Config.r')

###############################################
# Preqrequisites Configurations & Parameters

In [ ]:
### Load the parameters that are set via the configuration files

In [ ]:
### Load configurations file
global_configs = read.csv('configurations/Data_Configs.csv', sep = ',')

In [ ]:
head(global_configs,3)

In [ ]:
data_path = global_configs$value[global_configs$parameter == 'data_path']

In [ ]:
data_path

In [ ]:
result_path = global_configs$value[global_configs$parameter == 'result_path']

In [ ]:
result_path

In [ ]:
## Load the configuration file specifying single-cell specific filtering options

In [ ]:
sc_configs = read.csv('configurations/02_Pre_Processing_Configs_SC.csv', sep = ',')

In [ ]:
head(sc_configs,2)

In [ ]:
sc_configs = sc_configs[sc_configs$data_name != '',]

In [ ]:
sc_configs

In [ ]:
## Load the configuration file specifying the pre-processing options for all datasets

In [ ]:
data_configs = read.csv('configurations/02_Pre_Processing_Configs.csv', sep = ',')

In [ ]:
data_configs = data_configs[data_configs$configuration_name != '',]   # remove lines with empty configuration names
data_configs = data_configs[!is.na(data_configs$configuration_name),]  # remove lines with NA in configuration name

In [ ]:
head(data_configs)

In [ ]:
### Generate the result data directory if it does not exist yet
if(!file.exists(paste0(result_path, '02_results'))){
    dir.create(file.path(paste0(result_path, '02_results')))
    }

# Load Data

In [ ]:
### Load sc Data and exclude cluster_ids as specified in the configuration file

In [ ]:
datasets = list()

In [ ]:
## Load sc data (pseudobulk) generated in previous step
if(nrow(sc_configs) > 0) {
  for(j in 1:nrow(sc_configs)) {
    sc_data_name = sc_configs$data_name[j]
    current_config = sc_configs$configuration_name[j]
    
    file_path = paste0(result_path, '/01_results/01_', sc_data_name, 'Pseudobulk_Table', '.csv')
    
    if(file.exists(file_path)) {
      sc_data = fread(file_path)
      sc_data$V1 = NULL
      
      ## Filter the sc_data for the current data_name only
      data = sc_data[sc_data$dataset == sc_data_name, ]
      
      ## Exclude cluster_id's (cell-type clusters) if needed
      if(!is.na(sc_configs$cell_type_exclusion[j])) {
        data = data[!data$type %in% unlist(strsplit(sc_configs$cell_type_exclusion[j], ',')), ]
      }
      
      # Store data into the datasets list using the config name and data name
      datasets[[current_config]][[sc_data_name]] = data
      
      popup_function_pos('Single Cell data Loaded')
    } else {
      print('No single-cell data loaded: check whether 01_Prepare_Pseudobulk.ipynb has been executed successfully ')
      popup_function_neg('No single-cell data loaded: check whether 01_Prepare_Pseudobulk.ipynb has been executed successfully ')
    }  
  }
}

In [ ]:
### Load the other datasets specified in the configuration file

In [ ]:
for(i in unique(data_configs$configuration_name)){     # for each config
    for(j in unique(data_configs$data_name[data_configs$configuration_name == i])){      # each specifiec data-name
        
        configuration = data_configs[(data_configs$configuration_name == i) & (data_configs$data_name == j),]
        
        if(configuration$file_type == 'csv' | configuration$file_type == '.csv'){
            file_path = paste0(data_path, j, '.csv')
            if(file.exists(file_path)){
            
                data = read.csv(paste0(data_path, j, '.csv'))
                data$X = NULL
                data = melt(data, id.vars = 'sample_id')
                data$dataset = j
                data$type = configuration$data_type

                datasets[[i]][[j]] = data
                popup_function_pos(paste0('Data ', j, ' has been loaded successfully'))
                
                }
               else{ popup_function_neg(paste0('Error: Data ', j, ' could not be loaded. Check the data name and file path'))}
               
            
        }
        }
    }

In [ ]:
data_backup = datasets # in case something should be re-executed, so loading of data is not necessary a second time

In [ ]:
datasets = data_backup

In [ ]:
#str(datasets)

# Pre-Process each dataset as specified in the configuration files

## Sample Filter

In [ ]:
### Filter out sample_id's specified in the configuration file

In [ ]:
head(data_configs,3)

In [ ]:
for(i in 1:nrow(data_configs)){
    ### Remove samples based on specified samples in remove_sample_ids column
    if( (!is.na(data_configs$remove_sample_ids[i])) & (data_configs$remove_sample_ids[i] != '')){
        
        print(paste0('Filtered specific samples for ',data_configs$data_name[i], ' ',  unique( unlist(strsplit(data_configs$remove_sample_ids[i], ',')))))
        
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        ### remove samples
        data = data[! data$sample_id %in% unlist(strsplit(data_configs$remove_sample_ids[i], ',')),]  # TBD check!
        
        ### replace adjusted data
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data
        
        }
    
     ### Remove samples based on threshold in sample_filtering_thres
     if ( (as.numeric(data_configs$sample_filtering_thres[i]) < 1) & (as.numeric(data_configs$sample_filtering_thres[i]) > 0)){
         
         print(paste0('Filtered samples based on threshold for ',data_configs$data_name[i])) 
         data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
         print(paste0('Amount samples before filtering ', length(unique(data$sample_id))))
         
         ### calculate percentage of features with zero values
         data = data %>% group_by(sample_id, type) %>% mutate(zero_expression_percentage = sum(value == 0)/ n())
         ### filter out samples if percentage higher than threshold
         data = data[data$zero_expression_percentage < data_configs$sample_filtering_thres[i],]
         print(paste0('Amount samples after filtering ', length(unique(data$sample_id))))
         popup_function_pos(paste0('Filtered samples for: ', data_configs$data_name[i], ' Amount samples after filtering ', length(unique(data$sample_id))))
         
         
         ### remove generated columns
         data$zero_expression_percentage = NULL
         
         ### replace adjusted data
         datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data            
    } 
}
        

In [ ]:
#datasets

## Feature Removal (based on sample expression)

In [ ]:
## Filter out features that are not expressed in a certain amount of sample (threshold set in the configuration file)

In [ ]:
head(data_configs,3)

In [ ]:
data_configs$configuration_name[i]

In [ ]:
data_configs$data_name[i]

In [ ]:
data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]

In [ ]:
data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]

In [ ]:
data

In [ ]:
head(data,2)

In [ ]:
for(i in 1:nrow(data_configs)){

    if( (!is.na(data_configs$feature_filtering_thres[i])) & (data_configs$feature_filtering_thres[i] != '')  & (data_configs$feature_filtering_thres[i] > 0)){
        
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        print(paste0(data_configs$configuration_name[i], ' ' ,data_configs$data_name[i]))
        
        ## Determine data to filter
        data$expression = TRUE
        data$expression[data$value == 0] = FALSE
        expression_filter = data %>% group_by(type, variable) %>% summarise(perc_expression = sum(expression)  )
        expression_filter$perc_expression = expression_filter$perc_expression / length(unique(data$sample_id))
        
        ## Apply filter
        filtered_out = expression_filter[expression_filter$perc_expression <= data_configs$feature_filtering_thres[i],]
        print(paste0( 'Filtered: ' ))
        if(nrow(filtered_out) > 0){
            print((head(filtered_out %>% dplyr::group_by(type) %>% dplyr::count())))
            }
        expression_filter = expression_filter[expression_filter$perc_expression >data_configs$feature_filtering_thres[i],]  # kept data
        
        data = merge(data, expression_filter[,c('type', 'variable')], by.x = c('type', 'variable'), by.y = c('type', 'variable'))   # filter the data
        
        ## Remove expression column 
        data$expression = NULL
        
        ## Replace 
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]  = data
        
      }
}

In [ ]:
names(datasets)

## Library Adjustment

In [ ]:
## Normalize measured counts for each sample to have the same amount of counts

In [ ]:
head(data_configs,3)

In [ ]:
for(i in 1:nrow(data_configs)){
    if((data_configs$library_adjustment[i] == 'TRUE')){
        
        print(paste0('Library Adjustment for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Library Adjustment for ',data_configs$data_name[i]))
        
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]

        ### Calculate scaling factor per sample
        data = data %>% group_by(sample_id,type) %>% mutate(sample_counts = sum(value))
        data = data %>% group_by(type) %>% mutate(mean_sample_counts = mean(sample_counts))
        
        data$scaling_factor = data$sample_counts/ data$mean_sample_counts
        data$scaling_factor[data$scaling_factor == 0] = 1 # avoid dividing by 0; TBD whether to include or exclude samples with only zero counts in a cell-type
        
        ### Apply scaling to counts
        
        data$value = data$value / data$scaling_factor
        
        ### Save transformed data to list
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data
        
        }
    }
        

In [ ]:
names(datasets)

## Gene Filtering (according to cells expressing genes - only for sc Data)

In [ ]:
### Remove genes from the single-cell dataset that are expressed in a too low amount of cells

In [ ]:
head(sc_configs,2)

In [ ]:
## Load gene filtering information from previous script

In [ ]:
gene_expression_info = data.frame()

In [ ]:
for(i in sc_configs$data_name){
    data= read.csv(paste0(result_path, '/01_results/01_' ,i, '_Gene_Expr_per_Cell_Type.csv'))
    data$X = NULL
    
    data$data_name = i
    gene_expression_info = rbind(gene_expression_info, data)
    }

In [ ]:
head(gene_expression_info,2)

In [ ]:
sc_configs

In [ ]:
if(nrow(sc_configs) > 0){
for(i in unique(gene_expression_info$data_name)){
        
        print(paste0('Gene Filtering for  ',sc_configs$configuration_name[sc_configs$data_name == i]))
        popup_function_pos(paste0('Gene Filtering for  ', i))
        
        data = datasets[[sc_configs$configuration_name[sc_configs$data_name == i]]][[i]]
        gene_expr_data = gene_expression_info[gene_expression_info$data_name == i,]
    
        ## Get thresholds for config
        thres1 = as.numeric(unlist(str_split(sc_configs$cell_expr_thres1[sc_configs$data_name == i], ';')))
        thres2 = as.numeric(unlist(str_split(sc_configs$cell_expr_thres2[sc_configs$data_name == i], ';')))
        amount_samples = length(unique(data$sample_id))
        print(paste0('Amount Samples', amount_samples))
    
        ## Filter down gene based on the expression info
        gene_filtering =  gene_expr_data[((     gene_expr_data$perc_cells > thres1[1]) & (     gene_expr_data$total_amount_cells_expressing_gene > amount_samples * thres1[2])) |
         ((     gene_expr_data$perc_cells > thres2[1]) & (     gene_expr_data$total_amount_cells_expressing_gene > amount_samples * thres2[2])) ,]
    
        ## Apply to thresholds set in the configuration file
        filtered_data = data.frame()
        for( k in unique(data$type)){
            data_cluster = data[(data$type == k) & (data$variable %in% gene_filtering$gene[gene_filtering$cluster == k]),]
            filtered_data = rbind(filtered_data, data_cluster)
            }
        datasets[[sc_configs$configuration_name[sc_configs$data_name == i]]][[i]] = filtered_data
    }
    }

In [ ]:
### Amount of genes after filtering

In [ ]:
#unique(datasets[[1]][['Prepared_sc_Data']][,c('type', 'variable')])%>% group_by(type) %>% dplyr::count()

In [ ]:
head(filtered_data,2)

## Log Transformation

In [ ]:
## Apply log transformation to the data types specified in the configuration file

In [ ]:
head(data_configs,2)

In [ ]:
data_configs

In [ ]:
for(i in 1:nrow(data_configs)){
    if((data_configs$log_transformation[i] == 'TRUE')){
        
        print(paste0('Log Transformation for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Log Transformation for ',data_configs$data_name[i]))
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        data$value = log2(data$value + 1)  # add pseudocount of 1
        
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data # save adjusted data
        }
    }
        
        

## Variable Gene Filtering

In [ ]:
### Filter on highly variable genes if specified in the configuration file

In [ ]:
head(data_configs,2)

In [ ]:
for(i in 1:nrow(data_configs)){
    ### Filter genes with lowest variance
    if ( (as.numeric(data_configs$variable_genes_filtering[i]) < 1) & (as.numeric(data_configs$variable_genes_filtering[i]) > 0)){
        print(paste0('Variable Genes Filtering for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Variable Genes Filtering for ',data_configs$data_name[i]))
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        ### Calculate variance and threshold
        data = data %>% group_by(variable, type) %>% mutate(feature_variance = var(value)) # variance
        data = data %>% group_by(type) %>% mutate(variance_threshold = quantile(feature_variance, probs = seq(0, 1, 0.01), na.rm = FALSE,
         names = TRUE)[(1-as.numeric(data_configs$variable_genes_filtering[i]))*100])   # threshold
        
        ### Filter
        data = data[data$feature_variance > data$variance_threshold,]
        
        ### remove generated columns
        data$feature_variance = NULL
        data$variance_threshold = NULL
        
        ### Save transformed data to list
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data
        
        }
    }
        

## Sample Quantile Normalization

In [ ]:
### Apply Sample Quantile Normalization for the data-types specified in the configuration file

In [ ]:
head(data_configs,6)

In [ ]:
for(i in 1:nrow(data_configs)){
    if((data_configs$quantile_normalization_samples[i] == 'TRUE')){
        
        print(paste0('Sample Quantile Normalization for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Sample Quantile Normalization for ',data_configs$data_name[i]))
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        transformed_data = data.frame()
        
        for(k in unique(data$type)){
            data_type = data[data$type == k,]
            data_type = data_type %>% dcast(variable ~ sample_id, value.var = 'value')
            features = data_type$variable
            rownames(data_type) = features
            data_type$variable = NULL
            data_type = data_type[,colSums(is.na(data_type)) != nrow(data_type)] # remove na samples
            data_type  = quantile_normalization(data_type ) 
            data_type = data.frame(data_type)
            data_type$variable = features
            data_type = melt(data_type)
            colnames(data_type) = c('variable', 'sample_id', 'value')
            
            data_type$type = k 
            data_type$dataset = data_configs$data_name[i]
            transformed_data = rbind(transformed_data, data_type)
            
            }
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = transformed_data
        }
    }
            
        
        
        

## Gene Removal (ribosomal, mitochondrial)

In [ ]:
### Remove ribosomal and mitochondrial genes (only works if 'Gene' annotation is given as SYMBOL

In [ ]:
head(data_configs,2)

In [ ]:
for(i in 1:nrow(data_configs)){
    if((data_configs$ribosomal_mitochondrial_gene_filtering[i] == 'TRUE')){
        
        print(paste0('Remove ribosomal and mitochondrial genes for ',data_configs$data_name[i]))
        popup_function_pos(paste0('Remove ribosomal and mitochondrial genes for ',data_configs$data_name[i]))
        data = datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]]
        
        ## Remove ribosomal and mitochondiral genes
        data = data[is.na(str_extract(data$variable, '^MT.*|^RPL.*|^RPS.*')),]
        
        datasets[[data_configs$configuration_name[i]]][[data_configs$data_name[i]]] = data
        }
    }
        

        
        

# Merge data types and process

In [ ]:
### Combine all the datasets to one dataset

In [ ]:
#str(datasets)

In [ ]:
### Combine all types into one dataset
datasets = lapply(datasets, function(x){
    data = do.call(rbind, x)
    })

In [ ]:
### Overview amount of features per type/ view 

In [ ]:
unique(datasets[[1]][,c('type', 'variable')]) %>% group_by(type) %>% dplyr::count()

# Feature Wise Quantile Normalization

In [ ]:
### Apply feature wise quantile normalization if specified in the configuration file

In [ ]:
head(data_configs,6)

In [ ]:
data_configs$feature_wise_quantile_normalization

In [ ]:
names(datasets)

In [ ]:
data_configs

In [ ]:

for (i in names(datasets)) {
  data <- datasets[[i]]

  # keep the original identifier: cell-type + feature
  # so columns stay unique for AC-like, Astrocyte, ...
  data$ident <- paste0(data$type, "_0_", data$variable)

  # wide
  final_data <- dcast(data, sample_id ~ ident, value.var = "value")
  rownames(final_data) <- final_data$sample_id
  final_data$sample_id <- NULL

  # drop samples that are all NA
  data_nas <- is.na(final_data)
  keep_samples <- rownames(final_data)[rowSums(data_nas) != ncol(final_data)]
  final_data <- final_data[keep_samples, ]
  data_nas <- data_nas[keep_samples, ]

  # which datasets (views) should be normalized for this config?
  norm_views <- data_configs$data_name[
    data_configs$configuration_name == i &
      data_configs$feature_wise_quantile_normalization
  ]

  # build a lookup: cell_type -> dataset
  # this is from the *long* data, so it's correct
  celltype2dataset <- unique(data.frame(
    type    = data$type,
    dataset = data$dataset,
    stringsAsFactors = FALSE
  ))
  # in case of duplicates, keep first
  rownames(celltype2dataset) <- celltype2dataset$type

  # get column "types" = the part before _0_
  col_types <- sapply(strsplit(colnames(final_data), "_0_"), `[`, 1)

  # normalize column-by-column, but decide using the dataset behind that cell type
  for (j in seq_along(colnames(final_data))) {
    this_type <- col_types[j]

    # some columns might not be real data cols (unlikely, but be safe)
    if (this_type %in% rownames(celltype2dataset)) {
      this_dataset <- celltype2dataset[this_type, "dataset"]

      # if the dataset of this cell type is marked for normalization → normalize
      if (this_dataset %in% norm_views) {
        final_data[, j] <- stdnorm(final_data[, j])
      }
    }
  }

  # restore NA pattern and melt back
  final_data <- data.frame(final_data)
  final_data[data_nas] <- NA
  final_data$sample_id <- rownames(final_data)

  data_long <- melt(final_data, id.vars = "sample_id")
  data_long$type     <- str_extract(data_long$variable, ".*_0_")
  data_long$type     <- str_replace(data_long$type, "_0_", "")
  data_long$variable <- str_replace(data_long$variable, ".*_0_", "")

  datasets[[i]] <- data_long
}

In [ ]:
head(datasets[[1]],2)

# Save the data

In [ ]:
### Save the data to use as input in the next script

In [ ]:
### Adjust variable names
datasets = lapply(datasets, function(x){
    x$gene = x$variable
    x$variable = paste0(x$type, '__', x$variable)
    return(x)
    })

In [ ]:
for(i in names(datasets)){
    write.csv(datasets[[i]], paste0(result_path, '/02_results/02_Combined_Data_', i, '_INTEGRATED',  '.csv'))
    popup_function_pos(paste0('Saved', result_path, '/02_results/' , ' 02_Combined_Data_', i, '_INTEGRATED',  '.csv'))
    }

In [ ]:
### Example of structure of dataset

In [ ]:
head(datasets[[1]],2)

In [ ]:
length(unique(datasets[[1]]$sample_id))

In [ ]:
datasets[[1]]$sample_id

# Update Configuration File

In [ ]:
#paste0(unique(datasets[[i]]$type), collapse = ',')  # TBD make config specific

In [ ]:
### Adjust 06 Pathway Analysis Configs

In [ ]:
#configs06 = data.frame(
#    mofa_result_name = paste0(unique(data_configs$configuration_name), '_MOFA'),
#    factor_set = '1',
#    coverage_par = 0.2,
#    types = paste0(unique(datasets[[i]]$type), collapse = ','),
#    coverage_plot  = 0.5,
#    p_value_plot = 0.05,
#    max_pathways_plot = 8,
#    enrichment_plot = 'negative',
#    top_features_plot = 0.125,
#    pathway_selection = '')

In [ ]:
#configs06

In [ ]:
#write.csv(configs06, 'configurations/06_Pathway_Configs.csv', row.names = FALSE)

In [ ]:
### Inform about successful execution
popup_function_pos('02_Integrate_and_Normalize_Data_Sources: Execution Finished')

In [ ]:
Sys.sleep(20)
popup_function_info('02_Integrate_and_Normalize_Data_Sources')